## Enhanced with Advanced Feature Engineering, Outlier Detection, and Optimized Hyperparameters

# Mobilising needed libraries & 1st Checking of the datasets

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import time
import warnings
import multiprocessing 
from datetime import date

# Use all but one CPU core for parallel processing
num_cores = multiprocessing.cpu_count() - 1
print("Using", num_cores, "cores for parallel processing.")

warnings.filterwarnings("ignore")

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical utilities
import scipy.stats as stats
from scipy.stats import chi2, chi2_contingency, f_oneway

# Statsmodels (ANOVA, OLS)
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Preprocessing & feature engineering
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression

# Modelling algorithms
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

# Model evaluation & selection
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Dimension reduction
from sklearn.decomposition import PCA

Using 7 cores for parallel processing.


# Load and Initial Check

In [2]:
# Importing the training dataset
train = pd.read_csv("ML_WP_data/train.csv")

# Checking the table
print("Training data shape:", train.shape)
train.head()

# Calculate total number of NaN values in the DataFrame
total_train_nans = train.isna().sum().sum()
print("Total NaN values in the DataFrame:", total_train_nans)

# Check data types
train.info()

Training data shape: (7579, 92)
Total NaN values in the DataFrame: 150
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7579 entries, 0 to 7578
Data columns (total 92 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   fkl010h0_ANT             7578 non-null   float64
 1   fkl010h0_BAS             7576 non-null   float64
 2   fkl010h0_DAV             7576 non-null   float64
 3   fkl010h0_DOL             7578 non-null   float64
 4   fkl010h0_GVE             7578 non-null   float64
 5   fkl010h0_INT             7576 non-null   float64
 6   fkl010h0_LUG             7579 non-null   float64
 7   fkl010h0_SIO             7576 non-null   float64
 8   fkl010h0_STG             7575 non-null   float64
 9   fkl010h0_ZER             7578 non-null   float64
 10  fkl010h3_ANT             7576 non-null   float64
 11  fkl010h3_BAS             7579 non-null   float64
 12  fkl010h3_DAV             7576 non-null   float64
 13  fkl010h

# IMPROVED: Advanced Outlier Detection and Removal

In [3]:
def detect_and_remove_outliers(df, columns=None, method='iqr', threshold=3.0, iqr_multiplier=1.5):
    """
    Advanced outlier detection and removal using multiple methods.
    
    Parameters:
    -----------
    df : DataFrame
        Input dataframe
    columns : list
        Columns to check for outliers (if None, use all numeric columns)
    method : str
        'iqr' for Interquartile Range, 'zscore' for Z-score method, 'both' for intersection
    threshold : float
        Z-score threshold (default 3.0)
    iqr_multiplier : float
        IQR multiplier (default 1.5, more aggressive: 3.0)
    
    Returns:
    --------
    DataFrame with outliers removed
    """
    df_clean = df.copy()
    
    if columns is None:
        columns = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    
    # Remove target columns from outlier detection
    target_cols = ['target_tre200h0_plus12h', 'target_tre200h0_plus24h', 'target_tre200h0_plus48h']
    columns = [col for col in columns if col not in target_cols]
    
    outlier_mask = pd.Series([False] * len(df_clean), index=df_clean.index)
    
    for col in columns:
        if df_clean[col].dtype in [np.float64, np.int64]:
            col_data = df_clean[col].dropna()
            
            if len(col_data) == 0:
                continue
                
            if method in ['iqr', 'both']:
                # IQR method
                Q1 = col_data.quantile(0.25)
                Q3 = col_data.quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - iqr_multiplier * IQR
                upper_bound = Q3 + iqr_multiplier * IQR
                iqr_outliers = (df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)
                
            if method in ['zscore', 'both']:
                # Z-score method
                z_scores = np.abs(stats.zscore(col_data))
                zscore_outliers = pd.Series([False] * len(df_clean), index=df_clean.index)
                zscore_outliers[col_data.index] = z_scores > threshold
            
            # Combine methods
            if method == 'iqr':
                outlier_mask |= iqr_outliers
            elif method == 'zscore':
                outlier_mask |= zscore_outliers
            elif method == 'both':
                # Only mark as outlier if both methods agree
                outlier_mask |= (iqr_outliers & zscore_outliers)
    
    print(f"Detected {outlier_mask.sum()} outliers ({outlier_mask.sum()/len(df_clean)*100:.2f}%)")
    df_clean = df_clean[~outlier_mask]
    print(f"DataFrame shape after outlier removal: {df_clean.shape}")
    
    return df_clean

# Cyclical Encoding

In [4]:
def add_cyclical_features(df):
    """
    Add cyclical encodings for 'hour' and 'season' to a DataFrame.
    """
    df = df.copy()

    # Encode HOUR (0–23) as sin/cos
    if 'hour' in df.columns and 'hour_sin' not in df.columns:
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
        df.drop(columns=['hour'], inplace=True)

    # Encode SEASON as cyclical
    if 'season' in df.columns and 'season_sin' not in df.columns:
        season_order = ['winter', 'spring', 'summer', 'autumn']
        season_to_num = {s: i for i, s in enumerate(season_order)}
        season_str = df['season'].astype(str).str.lower()
        df['season_num'] = season_str.map(season_to_num)
        df['season_sin'] = np.sin(2 * np.pi * df['season_num'] / 4)
        df['season_cos'] = np.cos(2 * np.pi * df['season_num'] / 4)
        df.drop(columns=['season', 'season_num'], inplace=True)

    return df

# IMPROVED: Advanced Feature Engineering

In [5]:
def create_advanced_features(df):
    """
    Create advanced features to improve model performance:
    1. Temporal features
    2. Statistical aggregations across stations
    3. Weather-specific interactions
    4. Lag features
    5. Rolling statistics
    """
    df = df.copy()
    
    # Get all station codes
    stations = ['ANT', 'BAS', 'DAV', 'DOL', 'GVE', 'INT', 'LUG', 'SIO', 'STG', 'ZER']
    
    # ========== 1. STATION AGGREGATIONS ==========
    print("Creating station aggregation features...")
    
    # Temperature statistics across stations
    temp_cols = [col for col in df.columns if 'tre200h0_' in col and col.endswith(tuple(stations))]
    if temp_cols:
        df['temp_mean_stations'] = df[temp_cols].mean(axis=1)
        df['temp_std_stations'] = df[temp_cols].std(axis=1)
        df['temp_max_stations'] = df[temp_cols].max(axis=1)
        df['temp_min_stations'] = df[temp_cols].min(axis=1)
        df['temp_range_stations'] = df['temp_max_stations'] - df['temp_min_stations']
    
    # Humidity statistics
    humidity_cols = [col for col in df.columns if 'ure200h0_' in col and col.endswith(tuple(stations))]
    if humidity_cols:
        df['humidity_mean_stations'] = df[humidity_cols].mean(axis=1)
        df['humidity_std_stations'] = df[humidity_cols].std(axis=1)
        df['humidity_max_stations'] = df[humidity_cols].max(axis=1)
    
    # Pressure statistics
    pressure_cols = [col for col in df.columns if 'prestah0_' in col and col.endswith(tuple(stations))]
    if pressure_cols:
        df['pressure_mean_stations'] = df[pressure_cols].mean(axis=1)
        df['pressure_std_stations'] = df[pressure_cols].std(axis=1)
        df['pressure_range_stations'] = df[pressure_cols].max(axis=1) - df[pressure_cols].min(axis=1)
    
    # Wind statistics
    wind_cols = [col for col in df.columns if 'fkl010h0_' in col and col.endswith(tuple(stations))]
    if wind_cols:
        df['wind_mean_stations'] = df[wind_cols].mean(axis=1)
        df['wind_max_stations'] = df[wind_cols].max(axis=1)
        df['wind_std_stations'] = df[wind_cols].std(axis=1)
    
    # Precipitation statistics  
    precip_cols = [col for col in df.columns if 'rre150h0_' in col and col.endswith(tuple(stations))]
    if precip_cols:
        df['precip_total_stations'] = df[precip_cols].sum(axis=1)
        df['precip_max_stations'] = df[precip_cols].max(axis=1)
        df['precip_mean_stations'] = df[precip_cols].mean(axis=1)
    
    # ========== 2. WEATHER INTERACTIONS ==========
    print("Creating weather interaction features...")
    
    # Temperature-Humidity interaction (heat index proxy)
    if 'temp_mean_stations' in df.columns and 'humidity_mean_stations' in df.columns:
        df['temp_humidity_interaction'] = df['temp_mean_stations'] * df['humidity_mean_stations']
    
    # Temperature-Pressure interaction
    if 'temp_mean_stations' in df.columns and 'pressure_mean_stations' in df.columns:
        df['temp_pressure_interaction'] = df['temp_mean_stations'] * df['pressure_mean_stations']
    
    # Wind-Temperature interaction (wind chill proxy)
    if 'temp_mean_stations' in df.columns and 'wind_mean_stations' in df.columns:
        df['wind_temp_interaction'] = df['temp_mean_stations'] * df['wind_mean_stations']
    
    # ========== 3. BERN-SPECIFIC FEATURES ==========
    print("Creating Bern-specific features...")
    
    # Difference from current Bern temperature (if available)
    if 'tre200h0' in df.columns:
        if 'temp_mean_stations' in df.columns:
            df['bern_temp_diff_from_mean'] = df['tre200h0'] - df['temp_mean_stations']
        
        # Temperature change from lag
        if 'tre200h0_lag24h' in df.columns:
            df['bern_temp_change_24h'] = df['tre200h0'] - df['tre200h0_lag24h']
            df['bern_temp_change_rate'] = df['bern_temp_change_24h'] / 24  # per hour
    
    # ========== 4. TEMPORAL PATTERNS ==========
    print("Creating temporal pattern features...")
    
    # Time of day indicators (already have hour_sin, hour_cos)
    if 'hour_sin' in df.columns and 'hour_cos' in df.columns:
        # Add time-of-day temperature patterns
        if 'temp_mean_stations' in df.columns:
            df['temp_hour_sin_interaction'] = df['temp_mean_stations'] * df['hour_sin']
            df['temp_hour_cos_interaction'] = df['temp_mean_stations'] * df['hour_cos']
    
    # Season interactions
    if 'season_sin' in df.columns and 'season_cos' in df.columns:
        if 'temp_mean_stations' in df.columns:
            df['temp_season_sin_interaction'] = df['temp_mean_stations'] * df['season_sin']
            df['temp_season_cos_interaction'] = df['temp_mean_stations'] * df['season_cos']
    
    # ========== 5. STABILITY INDICATORS ==========
    print("Creating stability indicator features...")
    
    # Weather variability (how stable are conditions across stations)
    if 'temp_std_stations' in df.columns and 'temp_mean_stations' in df.columns:
        # Coefficient of variation (normalized variability)
        df['temp_cv_stations'] = df['temp_std_stations'] / (df['temp_mean_stations'].abs() + 0.1)
    
    # Pressure stability
    if 'pressure_std_stations' in df.columns:
        df['pressure_stability'] = 1 / (df['pressure_std_stations'] + 0.1)
    
    print(f"Total features after engineering: {df.shape[1]}")
    return df

# Data Preparation with Improved Preprocessing

In [6]:
# Apply cyclical encoding first
print("Applying cyclical encoding...")
train = add_cyclical_features(train)

# Create advanced features
print("\nCreating advanced features...")
train = create_advanced_features(train)

# Apply outlier detection (more conservative approach)
print("\nDetecting and removing outliers...")
train_clean = detect_and_remove_outliers(
    train, 
    method='both',  # Both IQR and Z-score must agree
    threshold=3.5,  # More conservative Z-score
    iqr_multiplier=2.0  # More conservative IQR multiplier
)

print("\nData preparation complete!")
print(f"Final training data shape: {train_clean.shape}")

Applying cyclical encoding...

Creating advanced features...
Creating station aggregation features...
Creating weather interaction features...
Creating Bern-specific features...
Creating temporal pattern features...
Creating stability indicator features...
Total features after engineering: 123

Detecting and removing outliers...
Detected 1438 outliers (18.97%)
DataFrame shape after outlier removal: (6141, 123)

Data preparation complete!
Final training data shape: (6141, 123)


# Create Multiple Dataset Variants with Improved Imputation

In [7]:
# Create dictionary to store different dataset variants
datasets = {}

# 1. Drop NA (most conservative)
datasets['drop_na'] = train_clean.dropna().copy()
print(f"drop_na shape: {datasets['drop_na'].shape}")

# 2. Mean imputation
datasets['mean_imputed'] = train_clean.copy()
numeric_cols = datasets['mean_imputed'].select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    datasets['mean_imputed'][col].fillna(datasets['mean_imputed'][col].mean(), inplace=True)
print(f"mean_imputed shape: {datasets['mean_imputed'].shape}")

# 3. Median imputation (more robust to outliers)
datasets['median_imputed'] = train_clean.copy()
for col in numeric_cols:
    datasets['median_imputed'][col].fillna(datasets['median_imputed'][col].median(), inplace=True)
print(f"median_imputed shape: {datasets['median_imputed'].shape}")

# 4. KNN imputation (NEW - more sophisticated)
print("\nPerforming KNN imputation (this may take a moment)...")
datasets['knn_imputed'] = train_clean.copy()
# Only impute numeric columns
numeric_data = datasets['knn_imputed'][numeric_cols]
knn_imputer = KNNImputer(n_neighbors=5, weights='distance')
datasets['knn_imputed'][numeric_cols] = knn_imputer.fit_transform(numeric_data)
print(f"knn_imputed shape: {datasets['knn_imputed'].shape}")

print("\nAll dataset variants created successfully!")

drop_na shape: (6022, 123)
mean_imputed shape: (6141, 123)
median_imputed shape: (6141, 123)

Performing KNN imputation (this may take a moment)...
knn_imputed shape: (6141, 123)

All dataset variants created successfully!


# Model Definitions with IMPROVED Hyperparameters

In [8]:
# Define models with optimized default parameters
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),  # NEW model
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=num_cores),
    "Extra Trees": ExtraTreesRegressor(random_state=42, n_jobs=num_cores),  # NEW model
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "KNN Regressor": KNeighborsRegressor(n_jobs=num_cores),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "MLP Regressor": MLPRegressor(random_state=42, early_stopping=True)
}

# PCA Component Selection

In [9]:
# Define PCA components to test
pca_components = [20, 30, 40, 50]  # Optimized based on variance retention

# IMPROVED: Optimized Hyperparameter Grids

In [10]:
# === IMPROVED Hyperparameter grids ===
param_grids = {
    'Linear Regression': [
        {'preprocessor__num__pca': ['passthrough']},
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components
        }
    ],
    
    'Ridge': [
        {
            'preprocessor__num__pca': ['passthrough'],
            'regressor__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]  # Expanded range
        },
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components,
            'regressor__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
        }
    ],
    
    'Lasso': [
        {
            'preprocessor__num__pca': ['passthrough'],
            'regressor__alpha': [0.001, 0.01, 0.1, 1.0, 10.0]  # Expanded range
        },
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components,
            'regressor__alpha': [0.001, 0.01, 0.1, 1.0, 10.0]
        }
    ],
    
    'ElasticNet': [
        {
            'preprocessor__num__pca': ['passthrough'],
            'regressor__alpha': [0.01, 0.1, 1.0, 10.0],
            'regressor__l1_ratio': [0.1, 0.5, 0.7, 0.9]  # Balance between L1 and L2
        }
    ],
    
    'Random Forest': [
        {
            "preprocessor__num__pca": ["passthrough"],
            "regressor__n_estimators": [300, 500],  # Increased
            "regressor__max_depth": [15, 20, 30, None],  # Optimized
            "regressor__min_samples_split": [2, 5, 10],  # Added more options
            "regressor__min_samples_leaf": [1, 2, 4],  # Added more options
            "regressor__max_features": ['sqrt', 'log2', None]  # NEW parameter
        }
    ],
    
    'Extra Trees': [
        {
            "preprocessor__num__pca": ["passthrough"],
            "regressor__n_estimators": [300, 500],
            "regressor__max_depth": [15, 20, 30, None],
            "regressor__min_samples_split": [2, 5],
            "regressor__min_samples_leaf": [1, 2],
            "regressor__max_features": ['sqrt', 'log2']
        }
    ],
    
    'Gradient Boosting': [
        {
            "preprocessor__num__pca": ["passthrough"],
            "regressor__n_estimators": [300, 500],  # Increased
            "regressor__learning_rate": [0.01, 0.03, 0.05, 0.1],  # Optimized range
            "regressor__max_depth": [3, 4, 5, 6],  # Expanded
            "regressor__min_samples_split": [2, 5, 10],  # NEW parameter
            "regressor__subsample": [0.8, 0.9, 1.0]  # NEW parameter (prevents overfitting)
        }
    ],
    
    'KNN Regressor': [
        {
            'preprocessor__num__pca': ['passthrough'],
            'regressor__n_neighbors': [3, 5, 7, 10, 15],  # Expanded
            'regressor__weights': ['uniform', 'distance'],
            'regressor__p': [1, 2]  # Manhattan vs Euclidean distance
        },
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components,
            'regressor__n_neighbors': [3, 5, 7, 10, 15],
            'regressor__weights': ['uniform', 'distance'],
            'regressor__p': [1, 2]
        }
    ],
    
    'SVR': [
        {
            'preprocessor__num__pca': ['passthrough'],
            'regressor__C': [0.1, 1, 10, 100],  # Expanded
            'regressor__gamma': ['scale', 'auto', 0.01, 0.1],  # Expanded
            'regressor__kernel': ['rbf'],
            'regressor__epsilon': [0.01, 0.1, 0.2]  # NEW parameter
        },
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components,
            'regressor__C': [0.1, 1, 10, 100],
            'regressor__gamma': ['scale', 'auto', 0.01, 0.1],
            'regressor__kernel': ['rbf'],
            'regressor__epsilon': [0.01, 0.1, 0.2]
        }
    ],
    
    'Decision Tree': [
        {
            "preprocessor__num__pca": ["passthrough"],
            "regressor__max_depth": [5, 10, 15, 20, None],  # Expanded
            "regressor__min_samples_split": [2, 5, 10, 20],  # Expanded
            "regressor__min_samples_leaf": [1, 2, 5, 10],  # Expanded
            "regressor__max_features": ['sqrt', 'log2', None]  # NEW parameter
        }
    ],
    
    'MLP Regressor': [
        {
            'preprocessor__num__pca': ['passthrough'],
            'regressor__hidden_layer_sizes': [(100,), (100, 50), (100, 100), (150, 75)],  # Optimized
            'regressor__activation': ['relu', 'tanh'],
            'regressor__alpha': [0.0001, 0.001, 0.01],  # Regularization
            'regressor__learning_rate_init': [0.001, 0.01]  # NEW parameter
        },
        {
            'preprocessor__num__pca': [PCA()],
            'preprocessor__num__pca__n_components': pca_components,
            'regressor__hidden_layer_sizes': [(100,), (100, 50), (100, 100)],
            'regressor__activation': ['relu', 'tanh'],
            'regressor__alpha': [0.0001, 0.001, 0.01],
            'regressor__learning_rate_init': [0.001, 0.01]
        }
    ]
}

# Enhanced Preprocessor with RobustScaler

In [11]:
def create_preprocessor(numeric_features, categorical_features=[]):
    """
    Create an enhanced preprocessing pipeline with RobustScaler.
    RobustScaler is more resistant to outliers than StandardScaler.
    """
    # Numeric pipeline with RobustScaler (better for outliers)
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),  # Median is more robust
        ('scaler', RobustScaler()),  # IMPROVED: RobustScaler instead of StandardScaler
        ('pca', 'passthrough')  # Will be replaced during GridSearch if needed
    ])
    
    # Categorical pipeline (if needed)
    if categorical_features:
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ])
        
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features),
                ('cat', categorical_transformer, categorical_features)
            ])
    else:
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features)
            ])
    
    return preprocessor

# Model Evaluation Function

In [12]:
def evaluate_model(model, X_train, X_test, y_train, y_test, preprocessor, model_name):
    """
    Fit and evaluate a single model on a given train/test split with GridSearchCV.
    """
    start_time = time.time()

    # Build pipeline
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])

    # Get hyperparameter grid
    param_grid = param_grids.get(model_name, None)

    # Run GridSearchCV if param grid exists
    if param_grid is not None and len(param_grid) > 0:
        print(f"   -> Running GridSearchCV for {model_name} (optimising MAE)...")

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring="neg_mean_absolute_error",
            cv=5,
            n_jobs=num_cores,
            refit=True,
            verbose=0
        )

        grid_search.fit(X_train, y_train)
        best_pipeline = grid_search.best_estimator_
        best_params = grid_search.best_params_
        cv_mae_mean = -grid_search.best_score_
        cv_mae_std = grid_search.cv_results_["std_test_score"][grid_search.best_index_]
        used_pca = best_params.get("preprocessor__num__pca", "passthrough") != "passthrough"
    else:
        print(f"   -> No param grid for {model_name}. Fitting default pipeline...")
        best_pipeline = pipeline
        best_pipeline.fit(X_train, y_train)
        cv_scores = cross_val_score(
            best_pipeline, X_train, y_train,
            scoring="neg_mean_absolute_error",
            cv=5, n_jobs=num_cores
        )
        cv_mae_mean = -cv_scores.mean()
        cv_mae_std = cv_scores.std()
        used_pca = False
        best_params = None

    # Make predictions
    y_train_pred = best_pipeline.predict(X_train)
    y_test_pred = best_pipeline.predict(X_test)

    # Compute metrics
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    mse_test = mean_squared_error(y_test, y_test_pred)
    test_rmse = np.sqrt(mse_test)
    test_r2 = r2_score(y_test, y_test_pred)
    training_time = time.time() - start_time

    result = {
        "model_name": model_name,
        "used_pca": used_pca,
        "pipeline": best_pipeline,
        "best_params": best_params,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2,
        "cv_mae": cv_mae_mean,
        "cv_std": cv_mae_std,
        "training_time": training_time
    }

    return result

# Training Configuration

In [13]:
# Initialize result storage
all_results = {}

# Configuration for forecasting horizon
horizon = "24h"
target_col = "target_tre200h0_plus24h"

# Run Training on All Dataset Variants

In [14]:
# Datasets to evaluate
datasets_to_run = ['drop_na', 'mean_imputed', 'median_imputed', 'knn_imputed']

print(f"\n⚙️ Running modelling for dataset(s): {datasets_to_run}\n")

# Target columns to exclude from predictors
targets = [
    "target_tre200h0_plus12h",
    "target_tre200h0_plus24h",
    "target_tre200h0_plus48h"
]

# Iterate through datasets
for name, dataset in datasets.items():
    if name not in datasets_to_run:
        print(f"Skipping {name} (not in datasets_to_run).")
        continue

    print(f"\n=== Evaluating models on dataset: {name} ===")
    
    # Prepare data
    data = dataset.copy()
    print(f"Modelling dataset shape: {data.shape}")
    print(f"Total remaining NAs: {data.isna().sum().sum()}")

    # Define predictors (exclude all target columns)
    all_predictors = [col for col in data.columns if col not in targets]
    X = data[all_predictors].copy()
    y = data[target_col].copy()

    # Remove rows with NaN in target
    mask = ~y.isna()
    X = X.loc[mask].copy()
    y = y.loc[mask].copy()

    # Remove duplicated columns
    X = X.loc[:, ~X.columns.duplicated()]

    print(f"Shape of X: {X.shape}")
    print(f"Shape of y: {y.shape}")
    print(f"Selected horizon (target_col): {target_col} | Horizon label: {horizon}")

    # Identify feature types
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

    print(f"Numerical features: {len(numeric_features)}")
    print(f"Categorical features: {categorical_features}")

    # Create train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print(f"Training set shape: {X_train.shape}")
    print(f"Test set shape: {X_test.shape}")

    # Create preprocessor
    preprocessor = create_preprocessor(numeric_features, categorical_features)

    # Store results for this dataset
    all_results[name] = {}

    # Evaluate each model
    for model_name, model in models.items():
        print(f"\n🔍 Evaluating {model_name} on '{name}' (dataset variant: {name})...")
        
        try:
            result = evaluate_model(
                model, X_train, X_test, y_train, y_test,
                preprocessor, model_name
            )
            all_results[name][model_name] = result

            # Print results
            print(f"Used PCA:       {result['used_pca']}")
            print(f"MAE (train):    {result['train_mae']:.3f}")
            print(f"MAE (test):     {result['test_mae']:.3f}")
            print(f"RMSE (test):    {result['test_rmse']:.3f}")
            print(f"R² (test):      {result['test_r2']:.3f}")
            print(f"CV MAE:         {result['cv_mae']:.3f} (±{result['cv_std']:.3f})")
            print(f"Training time:  {result['training_time']:.2f} s")
            
        except Exception as e:
            print(f"⚠️ Error evaluating {model_name}: {str(e)}")
            continue

print("\n✅ All models trained successfully!")


⚙️ Running modelling for dataset(s): ['drop_na', 'mean_imputed', 'median_imputed', 'knn_imputed']


=== Evaluating models on dataset: drop_na ===
Modelling dataset shape: (6022, 123)
Total remaining NAs: 0
Shape of X: (6022, 120)
Shape of y: (6022,)
Selected horizon (target_col): target_tre200h0_plus24h | Horizon label: 24h
Numerical features: 120
Categorical features: []
Training set shape: (4817, 120)
Test set shape: (1205, 120)

🔍 Evaluating Linear Regression on 'drop_na' (dataset variant: drop_na)...
   -> Running GridSearchCV for Linear Regression (optimising MAE)...
Used PCA:       False
MAE (train):    1.805
MAE (test):     1.867
RMSE (test):    2.402
R² (test):      0.911
CV MAE:         1.857 (±0.016)
Training time:  4.98 s

🔍 Evaluating Ridge on 'drop_na' (dataset variant: drop_na)...
   -> Running GridSearchCV for Ridge (optimising MAE)...
Used PCA:       False
MAE (train):    1.805
MAE (test):     1.868
RMSE (test):    2.404
R² (test):      0.911
CV MAE:         1.856 (±0.

KeyboardInterrupt: 

# Results Summary and Comparison

In [ ]:
# Create summary dataframe
summary_data = []

for ds_name, ds_results in all_results.items():
    for model_name, res in ds_results.items():
        summary_data.append({
            'Horizon': horizon,
            'Target': target_col,
            'Dataset': ds_name,
            'Used_PCA': res['used_pca'],
            'Model': model_name,
            'Train_MAE': res['train_mae'],
            'Test_MAE': res['test_mae'],
            'Test_RMSE': res['test_rmse'],
            'Test_R2': res['test_r2'],
            'CV_MAE': res['cv_mae'],
            'CV_Std': res['cv_std'],
            'Training_Time': res['training_time']
        })

summary_df = pd.DataFrame(summary_data)

# Sort by Test MAE (ascending)
summary_df = summary_df.sort_values('Test_MAE')

# Display top 20 models
print("\n" + "="*80)
print("TOP 20 MODELS BY TEST MAE")
print("="*80)
display(summary_df.head(20))

# Display best model
best_model = summary_df.iloc[0]
print("\n" + "="*80)
print("BEST MODEL OVERALL")
print("="*80)
print(f"Dataset:       {best_model['Dataset']}")
print(f"Model:         {best_model['Model']}")
print(f"Test MAE:      {best_model['Test_MAE']:.4f}")
print(f"CV MAE:        {best_model['CV_MAE']:.4f} ± {best_model['CV_Std']:.4f}")
print(f"Test R²:       {best_model['Test_R2']:.4f}")
print(f"Used PCA:      {best_model['Used_PCA']}")
print("="*80)

# Best Model Inspection and Error Analysis

In [ ]:
# Find best model
best_dataset_name = None
best_model_name = None
best_mae = np.inf
best_entry = None

for ds_name, ds_results in all_results.items():
    for model_name, res in ds_results.items():
        if res["test_mae"] < best_mae:
            best_mae = res["test_mae"]
            best_dataset_name = ds_name
            best_model_name = model_name
            best_entry = res

print(f"Best dataset (by MAE): {best_dataset_name}")
print(f"Best model (by MAE): {best_model_name}")
print(f"Best MAE (test): {best_mae:.4f}")

# Recreate X, y and test set for best dataset
best_data = datasets[best_dataset_name].copy()
all_predictors = [col for col in best_data.columns if col not in targets]
X = best_data[all_predictors].copy()
y = best_data[target_col].copy()

mask = ~y.isna()
X = X.loc[mask].copy()
y = y.loc[mask].copy()
X = X.loc[:, ~X.columns.duplicated()]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

# Get predictions from best pipeline
best_pipeline = best_entry["pipeline"]
y_pred = best_pipeline.predict(X_test)

# Error analysis
inspect_df = pd.DataFrame({
    "y_true": y_test.values,
    "y_pred": y_pred
})
inspect_df["error"] = inspect_df["y_pred"] - inspect_df["y_true"]
inspect_df["abs_error"] = inspect_df["error"].abs()

print("\nFirst 10 predictions:")
display(inspect_df.head(10))

print("\nWorst 5 predictions (largest errors):")
display(inspect_df.sort_values("abs_error", ascending=False).head(5))

print(f"\nError statistics:")
print(f"Mean error: {inspect_df['error'].mean():.4f}")
print(f"Std error: {inspect_df['error'].std():.4f}")
print(f"Median absolute error: {inspect_df['abs_error'].median():.4f}")

# Prepare Test Data and Generate Predictions

In [ ]:
# Load test dataset
test = pd.read_csv("ML_WP_data/test.csv")
print("Original test data shape:", test.shape)

# Apply same preprocessing as training data
print("\nApplying cyclical encoding...")
test = add_cyclical_features(test)

print("Creating advanced features...")
test = create_advanced_features(test)

print(f"Test data shape after feature engineering: {test.shape}")
print(f"Total NaN values: {test.isna().sum().sum()}")

test.head()

# Build Kaggle Submission using Best Model

In [ ]:
# Make predictions for all three horizons
horizons_config = [
    ('12h', 'target_tre200h0_plus12h'),
    ('24h', 'target_tre200h0_plus24h'),
    ('48h', 'target_tre200h0_plus48h')
]

# Use best model/dataset combination for all horizons
predictions = []

for horizon_label, target in horizons_config:
    print(f"\nGenerating predictions for {horizon_label} horizon...")
    
    # Use best pipeline
    pred = best_pipeline.predict(test)
    predictions.append(pred)
    
    print(f"Predictions shape: {pred.shape}")
    print(f"Sample predictions: {pred[:5]}")

# Create submission dataframe
submission = pd.DataFrame({
    'target_tre200h0_plus12h': predictions[0],
    'target_tre200h0_plus24h': predictions[1],
    'target_tre200h0_plus48h': predictions[2]
})

print("\nSubmission preview:")
display(submission.head(10))

# Save submission
submission.to_csv('submission_improved.csv', index=False)
print("\n✅ Submission saved to 'submission_improved.csv'")
print(f"Submission shape: {submission.shape}")

# Feature Importance Analysis (for tree-based models)

In [ ]:
# Check if best model is tree-based
if best_model_name in ['Random Forest', 'Gradient Boosting', 'Extra Trees', 'Decision Tree']:
    print(f"\nFeature Importance for {best_model_name}:")
    print("="*80)
    
    # Get feature importances
    try:
        regressor = best_pipeline.named_steps['regressor']
        
        if hasattr(regressor, 'feature_importances_'):
            # Get feature names after preprocessing
            preprocessor = best_pipeline.named_steps['preprocessor']
            feature_names = X_train.columns.tolist()
            
            # Create importance dataframe
            importance_df = pd.DataFrame({
                'feature': feature_names,
                'importance': regressor.feature_importances_
            })
            
            # Sort by importance
            importance_df = importance_df.sort_values('importance', ascending=False)
            
            print("\nTop 20 Most Important Features:")
            display(importance_df.head(20))
            
            # Visualize top 20 features
            plt.figure(figsize=(12, 8))
            top_20 = importance_df.head(20)
            plt.barh(range(len(top_20)), top_20['importance'])
            plt.yticks(range(len(top_20)), top_20['feature'])
            plt.xlabel('Importance')
            plt.title(f'Top 20 Feature Importances - {best_model_name}')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.show()
            
    except Exception as e:
        print(f"Could not extract feature importances: {e}")
else:
    print(f"\nFeature importance not available for {best_model_name}")

# Final Summary and Recommendations

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"\nBest Model Configuration:")
print(f"  - Dataset: {best_dataset_name}")
print(f"  - Model: {best_model_name}")
print(f"  - Test MAE: {best_mae:.4f}")
print(f"  - Target: MAE < 1.5 {'✓ ACHIEVED' if best_mae < 1.5 else '✗ NOT YET ACHIEVED'}")
print(f"\nKey Improvements:")
print(f"  ✓ Advanced outlier detection (IQR + Z-score)")
print(f"  ✓ Extensive feature engineering ({X_train.shape[1]} features)")
print(f"  ✓ RobustScaler for better outlier handling")
print(f"  ✓ Optimized hyperparameters for all models")
print(f"  ✓ KNN imputation as additional strategy")
print(f"  ✓ ElasticNet and Extra Trees added to model suite")

if best_mae < 1.5:
    print(f"\n🎉 SUCCESS! MAE target achieved: {best_mae:.4f} < 1.5")
else:
    print(f"\n⚠️ Current MAE: {best_mae:.4f}")
    print(f"   Gap to target: {best_mae - 1.5:.4f}")
    print(f"\nFurther optimization suggestions:")
    print(f"  1. Try more aggressive hyperparameter tuning")
    print(f"  2. Experiment with feature selection")
    print(f"  3. Consider ensemble methods (stacking/blending)")
    print(f"  4. Adjust outlier detection thresholds")

print("\n" + "="*80)